In [13]:
import pandas as pd
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

In [14]:
df =pd.read_csv("data/amazon.csv")

In [15]:
df =  df[['user_id', 'product_id', 'rating']]

In [16]:
df['user_id'] = df['user_id'].str.split(',')
df = df.explode('user_id')
df['user_id'] = df['user_id'].str.strip()

In [17]:
df.head()

,user_id,product_id,rating
0,AG3D6O4STAQKAY2UVGEUV46KN35Q,B07JW9H4J1,4.2
0,AHMY5CWJMMK5BJRBBSNLYT3ONILA,B07JW9H4J1,4.2
0,AHCTC6ULH4XB6YHDY6PCH2R772LQ,B07JW9H4J1,4.2
0,AGYHHIERNXKA6P5T7CZLXKVPT7IQ,B07JW9H4J1,4.2
0,AG4OGOFWXJZTQ2HKYIOCOY3KXF2Q,B07JW9H4J1,4.2


In [18]:
user_counts = df['user_id'].value_counts()
active_users = user_counts[user_counts >= 2].index
df_filtered = df[df['user_id'].isin(active_users)]

In [19]:
# Keep products that have at least 5 ratings
product_counts = df_filtered['product_id'].value_counts()
popular_products = product_counts[product_counts >= 2].index
df_filtered = df_filtered[df_filtered['product_id'].isin(popular_products)]

print(f"Original dataset size: {len(df)}")
print(f"Filtered dataset size: {len(df_filtered)}")

Original dataset size: 11503
Filtered dataset size: 3524


In [20]:
df_filtered.head()

,user_id,product_id,rating
0,AG3D6O4STAQKAY2UVGEUV46KN35Q,B07JW9H4J1,4.2
0,AHMY5CWJMMK5BJRBBSNLYT3ONILA,B07JW9H4J1,4.2
0,AHCTC6ULH4XB6YHDY6PCH2R772LQ,B07JW9H4J1,4.2
0,AGYHHIERNXKA6P5T7CZLXKVPT7IQ,B07JW9H4J1,4.2
0,AG4OGOFWXJZTQ2HKYIOCOY3KXF2Q,B07JW9H4J1,4.2


In [21]:
reader = Reader(rating_scale=(1,5))
data = Dataset.load_from_df(df_filtered[['user_id', 'product_id', 'rating']], reader)
trainset, testset = train_test_split(data, test_size=0.2, random_state =42)
print("Dataset Successfully split for training!")

Dataset Successfully split for training!


In [22]:
model = SVD()
print("Training the model...")
model.fit(trainset)
prediction = model.test(testset)
rmse = accuracy.rmse(prediction)
print(f"Model RMSE: {rmse}")

Training the model...
RMSE: 0.1163
Model RMSE: 0.11634847126399983


In [23]:
# 1. Pick a user and a product from your dataset
# (Make sure these exact strings exist in your raw CSV!)
test_user = 'AG3D6O4STAQKAY2UVGEUV46KN35Q'               # Replace with a real user ID from your data
test_product = 'B07JW9H4J1'       # We'll use the ASIN you mentioned earlier

# 2. Ask the model to predict the rating
prediction = model.predict(uid=test_user, iid=test_product)

# 3. Print the results cleanly
print("--- Prediction Results ---")
print(f"User: {prediction.uid}")
print(f"Product: {prediction.iid}")
print(f"Estimated Rating: {prediction.est:.2f} stars")

--- Prediction Results ---
User: AG3D6O4STAQKAY2UVGEUV46KN35Q
Product: B07JW9H4J1
Estimated Rating: 4.21 stars


In [25]:
df.shape

(11503, 3)

In [26]:
df_filtered.shape

(3524, 3)